# NAICS-2 Classification with RBF SVM

Using 3072-dim embeddings (OpenAI text-embedding-3-large) with an RBF kernel SVM.

5-fold stratified cross-validation on 20 NAICS-2 sector classes.

In [5]:
import pandas as pd
import numpy as np
import json
import os
import time
from sklearn.svm import SVC
from sklearn.preprocessing import LabelEncoder, normalize
from sklearn.decomposition import PCA
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("Imports done.")

Imports done.


In [6]:
csv_path = '../../.ipynb_checkpoints/ExioNAICS_embeddings_large_full.csv'
npy_cache = csv_path.rsplit('.', 1)[0] + '.npy'

df = pd.read_csv(csv_path)
print(f"Shape: {df.shape}")

if os.path.exists(npy_cache):
    print(f"\nLoading cached embeddings from {npy_cache}")
    start = time.time()
    X = np.load(npy_cache)
    print(f"Done in {time.time()-start:.1f}s")
else:
    print("\nParsing embeddings...")
    start = time.time()
    X = np.array([json.loads(e) for e in df['embeddings']], dtype=np.float32)
    print(f"Done in {time.time()-start:.1f}s")
    np.save(npy_cache, X)

df['emb'] = list(X)
print(f"Embedding shape: {X.shape}")

Shape: (20535, 6)

Loading cached embeddings from ../../.ipynb_checkpoints/ExioNAICS_embeddings_large_full.npy
Done in 0.0s
Embedding shape: (20535, 3072)


In [7]:
df = df.drop_duplicates(subset=['Company Name']).reset_index(drop=True)
X = normalize(np.array(df['emb'].tolist()))

NAICS2_MAP = {31: 33, 32: 33, 44: 45, 48: 49}
df['NAICS_2_merged'] = df['NAICS_2 Code'].replace(NAICS2_MAP)

le2 = LabelEncoder()
y2 = le2.fit_transform(df['NAICS_2_merged'].astype(str))

print(f"Total samples: {len(df)}, Classes: {len(le2.classes_)}")

SEED = 192
N_FOLDS = 5
PCA_DIMS = 512
TEST_SIZE = 0.1

# Hold out test set FIRST — never touched until final evaluation
X_dev, X_test, y_dev, y_test = train_test_split(
    X, y2, test_size=TEST_SIZE, random_state=SEED, stratify=y2
)
print(f"Dev: {len(X_dev)}, Test: {len(X_test)} (held out)")

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
print(f"{N_FOLDS}-fold stratified CV on dev set, PCA to {PCA_DIMS} dims")

Total samples: 14140, Classes: 20
Dev: 12726, Test: 1414 (held out)
5-fold stratified CV on dev set, PCA to 512 dims


## Train RBF SVM (K-Fold CV)

In [8]:
TOP_K = [1, 3, 5]
fold_topk = {k: [] for k in TOP_K}
all_y_true, all_y_pred = [], []

total_start = time.time()
for fold, (train_idx, val_idx) in enumerate(skf.split(X_dev, y_dev)):
    fold_start = time.time()

    # PCA fit on train fold only (no data leakage)
    pca = PCA(n_components=PCA_DIMS, random_state=SEED)
    X_tr = pca.fit_transform(X_dev[train_idx])
    X_va = pca.transform(X_dev[val_idx])
    y_tr, y_va = y_dev[train_idx], y_dev[val_idx]

    model = SVC(kernel='rbf', C=10.0, class_weight='balanced', probability=True, random_state=SEED)
    model.fit(X_tr, y_tr)

    # Top-k accuracy
    proba = model.predict_proba(X_va)
    for k in TOP_K:
        top_k_preds = np.argsort(proba, axis=1)[:, -k:]
        correct = np.any(top_k_preds == y_va[:, None], axis=1).mean()
        fold_topk[k].append(correct)

    # Collect predictions for aggregated report
    all_y_true.extend(y_va)
    all_y_pred.extend(model.predict(X_va))

    print(f"  Fold {fold+1}/{N_FOLDS}: top-1 {fold_topk[1][-1]:.4f}  ({time.time()-fold_start:.1f}s)")

print(f"\nTotal time: {time.time()-total_start:.1f}s")
print(f"\nMean across {N_FOLDS} folds:")
for k in TOP_K:
    scores = fold_topk[k]
    print(f"  Top-{k}: {np.mean(scores):.4f} ± {np.std(scores):.4f}")

  Fold 1/5: top-1 0.7054  (82.3s)
  Fold 2/5: top-1 0.6888  (83.0s)
  Fold 3/5: top-1 0.6963  (79.8s)
  Fold 4/5: top-1 0.6912  (79.7s)
  Fold 5/5: top-1 0.6974  (81.4s)

Total time: 406.2s

Mean across 5 folds:
  Top-1: 0.6958 ± 0.0058
  Top-3: 0.8879 ± 0.0046
  Top-5: 0.9382 ± 0.0043


## Per-class Performance (Aggregated across folds)

In [9]:
all_y_true = np.array(all_y_true)
all_y_pred = np.array(all_y_pred)
print(classification_report(all_y_true, all_y_pred, target_names=le2.classes_))

              precision    recall  f1-score   support

          11       0.67      0.66      0.67       465
          21       0.64      0.67      0.66       309
          22       0.70      0.73      0.72       120
          23       0.59      0.62      0.61       433
          33       0.76      0.86      0.81      4524
          42       0.57      0.44      0.50       980
          45       0.63      0.53      0.58      1066
          49       0.76      0.77      0.77       585
          51       0.65      0.63      0.64       438
          52       0.73      0.81      0.77       501
          53       0.56      0.44      0.49       280
          54       0.56      0.54      0.55       609
          55       0.00      0.00      0.00        37
          56       0.54      0.42      0.47       425
          61       0.76      0.71      0.73       191
          62       0.69      0.83      0.75       473
          71       0.69      0.63      0.66       260
          72       0.60    

## Confusion Matrix (Aggregated across folds)

In [10]:
cm = confusion_matrix(all_y_true, all_y_pred)
cm_df = pd.DataFrame(cm, index=le2.classes_, columns=le2.classes_)
print("Confusion Matrix (rows=true, cols=predicted):")
print(cm_df)

Confusion Matrix (rows=true, cols=predicted):
     11   21  22   23    33   42   45   49   51   52   53   54  55   56   61  \
11  309    4   0    4    84   24    7    3    2    3    2    6   0    3    0   
21    2  207   7    9    61    6    3    1    2    0    2    6   0    1    0   
22    0    7  88    0    14    2    1    2    0    1    0    1   0    2    0   
23    1    7   4  267    80    8   13    1    1    1   15   12   0   14    0   
33   59   47   8   62  3913  142   95   17   39   25    9   42   0   16    2   
42   43   20   1    9   329  432   85    9    6    9    5   10   0    6    1   
45   19    7   3   12   253   76  570    9   13   10    9   11   0    6    6   
49    6    3   3    1    43   13   10  452    2    3    5    7   0   19    1   
51    2    1   0    2    46    9   10    2  276   11    5   35   0    9    3   
52    2    5   1    1    14    8    6    1    8  404    9   12   2    5    0   
53    0    2   0    7    31    2   22   21    6   14  123   14   0    4   

## Final Test Evaluation

Train on ALL dev data, evaluate once on the held-out test set.

In [11]:
print("Training final model on full dev set...")
start = time.time()

pca_final = PCA(n_components=PCA_DIMS, random_state=SEED)
X_dev_pca = pca_final.fit_transform(X_dev)
X_test_pca = pca_final.transform(X_test)

final_model = SVC(kernel='rbf', C=10.0, class_weight='balanced', probability=True, random_state=SEED)
final_model.fit(X_dev_pca, y_dev)
print(f"Done in {time.time()-start:.1f}s")

# Top-k on test
proba_test = final_model.predict_proba(X_test_pca)
for k in TOP_K:
    top_k_preds = np.argsort(proba_test, axis=1)[:, -k:]
    correct = np.any(top_k_preds == y_test[:, None], axis=1).mean()
    print(f"  Test Top-{k}: {correct:.4f}")

# Full report
y_test_pred = final_model.predict(X_test_pca)
print(f"\n{classification_report(y_test, y_test_pred, target_names=le2.classes_)}")

cm_test = confusion_matrix(y_test, y_test_pred)
cm_test_df = pd.DataFrame(cm_test, index=le2.classes_, columns=le2.classes_)
print("Test Confusion Matrix (rows=true, cols=predicted):")
print(cm_test_df)

Training final model on full dev set...
Done in 116.5s
  Test Top-1: 0.6867
  Test Top-3: 0.9038
  Test Top-5: 0.9562

              precision    recall  f1-score   support

          11       0.68      0.73      0.70        52
          21       0.67      0.76      0.71        34
          22       0.56      0.77      0.65        13
          23       0.64      0.56      0.60        48
          33       0.76      0.85      0.80       503
          42       0.45      0.39      0.41       109
          45       0.54      0.50      0.52       118
          49       0.77      0.78      0.78        65
          51       0.69      0.59      0.64        49
          52       0.74      0.77      0.75        56
          53       0.60      0.39      0.47        31
          54       0.55      0.53      0.54        68
          55       0.00      0.00      0.00         4
          56       0.56      0.30      0.39        47
          61       0.74      0.81      0.77        21
          62    

/Users/brian/Projects/ClimateTurtles-ExioNaics-LLM-emissions-estimation/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/brian/Projects/ClimateTurtles-ExioNaics-LLM-emissions-estimation/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/brian/Projects/ClimateTurtles-ExioNaics-LLM-emissions-estimation/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in lab